#  ObjectLens AI — Targeted Object Recognition System

**Model:** MobileNetV2 (Pre-trained · ImageNet-1K)  
**Framework:** TensorFlow / Keras + Streamlit  
**Author:** Bashir Ahmad | B.Tech CSE · MNNIT Allahabad  

---

### What this notebook does
1. Installs Streamlit + ngrok tunnel dependencies
2. Writes `app.py` to disk (the full Streamlit app)
3. Launches the app in the background
4. Opens a **public ngrok URL** you can share for demo / submission


## Installing Dependencies


In [1]:
#  Install extra packages
!pip install streamlit pyngrok --quiet

# Verify TF is present
import tensorflow as tf
print(f" TensorFlow  : {tf.__version__}")

import streamlit
print(f" Streamlit   : {streamlit.__version__}")

import pyngrok
print(f" pyngrok     : {pyngrok.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 134.6 MB/s eta 0:00:00
 TensorFlow  : 2.19.0
 Streamlit   : 1.56.0
 pyngrok     : 8.0.0


## Write `app.py` to Disk
The complete Streamlit application is written directly from this notebook cell, no manual upload needed.

In [2]:
app_code = '''
"""
╔══════════════════════════════════════════════════════════════╗
║       TARGETED OBJECT RECOGNITION SYSTEM                     ║
║       Built with MobileNetV2 + Streamlit                     ║
║       Author: Bashir Ahmad | MNNIT Allahabad CSE             ║
╚══════════════════════════════════════════════════════════════╝
"""

import streamlit as st
import numpy as np
from PIL import Image
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
import time

# ─────────────────────────────────────────────────────────────
# PAGE CONFIGURATION  (must be first Streamlit call)
# ─────────────────────────────────────────────────────────────
st.set_page_config(
    page_title="ObjectLens AI",
    page_icon="🔍",
    layout="centered",
    initial_sidebar_state="collapsed",
)

# ─────────────────────────────────────────────────────────────
# CUSTOM CSS  — dark, precise, professional UI
# ─────────────────────────────────────────────────────────────
st.markdown("""
<style>
@import url(\'https://fonts.googleapis.com/css2?family=Space+Mono:wght@400;700&family=DM+Sans:wght@300;400;500;600&display=swap\');
:root {
    --bg:#0b0f19;--surface:#131929;--border:#1e2d45;--accent:#3b82f6;
    --accent-glow:rgba(59,130,246,0.18);--success:#10b981;--danger:#ef4444;
    --warn:#f59e0b;--text:#e2e8f0;--muted:#64748b;
    --mono:\'Space Mono\',monospace;--sans:\'DM Sans\',sans-serif;
}
html,body,[class*="css"]{font-family:var(--sans);background-color:var(--bg);color:var(--text);}
.stApp{background:var(--bg);}
.block-container{padding:2rem 1.5rem 4rem;max-width:800px;}
#MainMenu,footer,header{visibility:hidden;}
.hero{text-align:center;padding:2.5rem 0 1.5rem;border-bottom:1px solid var(--border);margin-bottom:2rem;}
.hero-badge{display:inline-block;font-family:var(--mono);font-size:0.65rem;letter-spacing:0.18em;
    color:var(--accent);background:var(--accent-glow);border:1px solid var(--accent);
    border-radius:2px;padding:3px 10px;margin-bottom:1rem;text-transform:uppercase;}
.hero h1{font-family:var(--mono);font-size:2rem;font-weight:700;color:#fff;margin:0 0 0.4rem;letter-spacing:-0.02em;}
.hero p{color:var(--muted);font-size:0.9rem;margin:0;}
.chip-row{display:flex;gap:10px;justify-content:center;margin:1.4rem 0;flex-wrap:wrap;}
.chip{font-family:var(--mono);font-size:0.72rem;padding:5px 14px;border-radius:2px;
    border:1px solid var(--border);color:var(--muted);letter-spacing:0.08em;background:var(--surface);}
.chip.active{border-color:var(--accent);color:var(--accent);background:var(--accent-glow);}
.section-label{font-family:var(--mono);font-size:0.65rem;letter-spacing:0.15em;
    color:var(--muted);text-transform:uppercase;margin-bottom:0.5rem;}
.result-card{background:var(--surface);border:1px solid var(--border);border-radius:4px;
    padding:1.6rem 2rem;margin-top:1.5rem;}
.result-label{font-family:var(--mono);font-size:0.6rem;letter-spacing:0.18em;
    color:var(--muted);text-transform:uppercase;margin-bottom:0.3rem;}
.result-object{font-family:var(--mono);font-size:2rem;font-weight:700;color:#fff;margin-bottom:0.3rem;}
.result-object.success{color:var(--success);}
.result-object.danger{color:var(--danger);}
.conf-bar-bg{background:var(--border);border-radius:1px;height:4px;width:100%;margin:1rem 0;overflow:hidden;}
.conf-bar-fill{height:4px;border-radius:1px;}
.conf-bar-fill.success{background:var(--success);}
.conf-bar-fill.danger{background:var(--danger);}
.conf-bar-fill.warn{background:var(--warn);}
.topk-table{width:100%;border-collapse:collapse;margin-top:1rem;font-size:0.78rem;}
.topk-table th{font-family:var(--mono);font-size:0.6rem;letter-spacing:0.15em;color:var(--muted);
    text-transform:uppercase;text-align:left;padding:6px 8px;border-bottom:1px solid var(--border);}
.topk-table td{padding:7px 8px;border-bottom:1px solid var(--border);color:var(--text);
    font-family:var(--mono);font-size:0.72rem;}
.topk-table tr:last-child td{border-bottom:none;}
.match-row td{color:var(--success) !important;}
.pill-row{display:flex;gap:8px;flex-wrap:wrap;margin-top:1.2rem;}
.pill{font-family:var(--mono);font-size:0.65rem;letter-spacing:0.08em;padding:4px 10px;
    border-radius:2px;background:var(--border);color:var(--muted);}
.stButton > button{font-family:var(--mono) !important;font-size:0.75rem !important;
    letter-spacing:0.12em !important;background:var(--accent) !important;color:#fff !important;
    border:none !important;border-radius:2px !important;padding:0.55rem 1.4rem !important;
    text-transform:uppercase !important;}
</style>
""", unsafe_allow_html=True)


# ─────────────────────────────────────────────────────────────
# PHASE A — Load MobileNetV2 CNN Brain
# ─────────────────────────────────────────────────────────────
@st.cache_resource(show_spinner=False)
def load_model():
    """
    Load MobileNetV2 with include_top=True so we get the full
    1000-class Softmax head pre-trained on ImageNet.
    """
    return MobileNetV2(weights="imagenet", include_top=True)


# ─────────────────────────────────────────────────────────────
# PHASE C — Target Filter Dictionary
# ─────────────────────────────────────────────────────────────
TARGET_KEYWORDS = {
    "Mobile Phone": [
        "cellular_telephone", "cellular_phone", "cell_phone",
        "mobile_phone", "iphone", "smartphone", "phone",
        "iPod", "remote_control",
    ],
    "Charger": [
        "power_strip", "electric_outlet", "plug", "adapter",
        "power_cable", "extension_cord", "switch", "transformer",
        "modem", "router", "hard_disc",
    ],
    "Pen": [
        "pen", "ballpoint", "fountain_pen", "pencil",
        "paintbrush", "lipstick", "drumstick",
        "knitting_needle", "nail", "skewer",
    ],
}

KEYWORD_MAP = {
    kw: target
    for target, keywords in TARGET_KEYWORDS.items()
    for kw in keywords
}


# ─────────────────────────────────────────────────────────────
# PHASE B — Preprocessing Pipeline
# ─────────────────────────────────────────────────────────────
def preprocess_image(image):
    """
    1. Convert to RGB  → handles RGBA / grayscale
    2. Resize to 224x224  → matches MobileNetV2 input layer
    3. Cast to float32 array  → shape (224, 224, 3)
    4. expand_dims(axis=0)  → shape (1, 224, 224, 3)  [batch dimension]
    5. preprocess_input  → normalizes pixels to [-1, 1]
    """
    img = image.convert("RGB").resize((224, 224))
    arr = np.array(img, dtype=np.float32)      # (224, 224, 3)
    arr = np.expand_dims(arr, axis=0)          # (1, 224, 224, 3)
    arr = preprocess_input(arr)                # normalize to [-1, 1]
    return arr


def run_inference(model, tensor, top_k=20):
    preds   = model.predict(tensor, verbose=0)
    decoded = decode_predictions(preds, top=top_k)[0]

    best_target, best_score, top_k_results = None, 0.0, []

    for _, label, score in decoded:
        top_k_results.append((label, float(score)))
        if best_target is None:
            for keyword, target_name in KEYWORD_MAP.items():
                if keyword.lower() in label.lower():
                    best_target = target_name
                    best_score  = float(score)
                    break

    return top_k_results, best_target, best_score


# ─────────────────────────────────────────────────────────────
# RENDER HELPERS
# ─────────────────────────────────────────────────────────────
def render_result_card(top_k_results, target, score):
    detected  = target is not None
    color_cls = "success" if detected else "danger"
    obj_text  = target if detected else "UNKNOWN"
    pct       = round(score * 100, 1) if detected else 0
    bar_cls   = "success" if pct >= 70 else ("warn" if pct >= 40 else "danger")

    st.markdown(f"""
    <div class="result-card">
        <div class="result-label">Detection Result</div>
        <div class="result-object {color_cls}">{obj_text}</div>
        <div style="font-size:0.8rem;color:var(--muted);margin-bottom:0.8rem;">
            {"Matched in top-20 ImageNet predictions" if detected
             else "No target class found in top-20 predictions"}
        </div>
        <div class="result-label">Confidence</div>
        <div style="font-family:var(--mono);font-size:1.3rem;color:#fff;">{pct}%</div>
        <div class="conf-bar-bg">
            <div class="conf-bar-fill {bar_cls}" style="width:{pct}%;"></div>
        </div>
        <div class="pill-row">
            <span class="pill">MODEL · MobileNetV2</span>
            <span class="pill">DATASET · ImageNet-1K</span>
            <span class="pill">TOP-K · 20</span>
            <span class="pill">INPUT · 224×224×3</span>
        </div>
    </div>
    """, unsafe_allow_html=True)

    if not detected:
        st.markdown("""
        <div style="margin-top:1rem;padding:1rem;border:1px solid var(--danger);
                    border-radius:4px;background:rgba(239,68,68,0.07);">
            <span style="font-family:var(--mono);font-size:0.72rem;
                         letter-spacing:0.12em;color:var(--danger);">
                ⚠ OOD OBJECT DETECTED — This object does not belong to any target class.
                The system refuses to make an unsupported classification.
            </span>
        </div>
        """, unsafe_allow_html=True)

    with st.expander("▸ Top-20 Raw Predictions (Softmax Scores)", expanded=False):
        rows = ""
        for label, prob in top_k_results[:20]:
            is_match = any(kw.lower() in label.lower() for kw in KEYWORD_MAP)
            row_cls  = \'class="match-row"\' if is_match else ""
            rows += f"""
            <tr {row_cls}>
                <td>{label}</td>
                <td>{prob*100:.3f}%</td>
                <td>{"✓ MATCH" if is_match else "—"}</td>
            </tr>"""
        st.markdown(f"""
        <table class="topk-table">
            <thead><tr>
                <th>ImageNet Label</th><th>Score</th><th>Filter Hit</th>
            </tr></thead>
            <tbody>{rows}</tbody>
        </table>
        """, unsafe_allow_html=True)


# ─────────────────────────────────────────────────────────────
# PHASE D — Main Streamlit App
# ─────────────────────────────────────────────────────────────
def main():
    st.markdown("""
    <div class="hero">
        <div class="hero-badge">Deep CNN · MobileNetV2 · ImageNet</div>
        <h1>ObjectLens AI</h1>
        <p>Domain-specific object recognition with Out-of-Distribution filtering</p>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("""
    <div class="chip-row">
        <span class="chip active">📱 Mobile Phone</span>
        <span class="chip active">🔌 Charger</span>
        <span class="chip active">🖊 Pen</span>
        <span class="chip">❌ Other → Unknown</span>
    </div>
    """, unsafe_allow_html=True)

    with st.spinner("Initialising MobileNetV2 weights..."):
        model = load_model()

    st.markdown("---")

    tab1, tab2 = st.tabs(["📷  Camera", "📁  Upload"])
    image_input = None

    with tab1:
        st.markdown(\'<div class="section-label">Live Camera Capture</div>\', unsafe_allow_html=True)
        camera_frame = st.camera_input("Point your camera at a Mobile Phone, Charger, or Pen")
        if camera_frame:
            image_input = Image.open(camera_frame)

    with tab2:
        st.markdown(\'<div class="section-label">Upload Image File</div>\', unsafe_allow_html=True)
        uploaded = st.file_uploader(
            "Drag & drop or browse",
            type=["jpg", "jpeg", "png", "webp"],
            label_visibility="collapsed",
        )
        if uploaded:
            image_input = Image.open(uploaded)

    if image_input is not None:
        col1, col2 = st.columns([1, 1], gap="medium")
        with col1:
            st.image(image_input, caption="Input Image", use_container_width=True)
        with col2:
            with st.spinner("Running inference..."):
                t0 = time.perf_counter()
                tensor = preprocess_image(image_input)
                top_k_results, target, score = run_inference(model, tensor)
                latency = (time.perf_counter() - t0) * 1000
            st.metric("Inference Time", f"{latency:.0f} ms")
        render_result_card(top_k_results, target, score)
    else:
        st.markdown("""
        <div style="text-align:center;padding:3rem 0;color:var(--muted);
                    font-family:var(--mono);font-size:0.75rem;
                    letter-spacing:0.12em;border:1px dashed var(--border);
                    border-radius:4px;margin-top:1.5rem;">
            AWAITING IMAGE INPUT
        </div>
        """, unsafe_allow_html=True)

    with st.sidebar:
        st.markdown("### 🧠 Model Info")
        st.json({
            "architecture": "MobileNetV2",
            "pretrained_on": "ImageNet-1K",
            "input_shape": [224, 224, 3],
            "top_k_search": 20,
            "normalization": "[-1, 1]",
            "output_classes": 1000,
        })
        st.markdown("### 🎯 Target Classes")
        for cls, kws in TARGET_KEYWORDS.items():
            st.markdown(f"**{cls}**: `{\'`, `\'.join(kws[:4])}`…")
        st.markdown("---")
        st.markdown(
            "<small style=\'color:#475569;font-family:monospace\'>"
            "Built by Bashir Ahmad · MNNIT Allahabad</small>",
            unsafe_allow_html=True,
        )


if __name__ == "__main__":
    main()
'''

with open('app.py', 'w') as f:
    f.write(app_code)

print(" app.py written successfully!")
print(f"   Size: {len(app_code)} characters")

 app.py written successfully!
   Size: 13415 characters


## Verify the Model Loads Correctly

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions

print("Loading MobileNetV2 ...")
model = MobileNetV2(weights='imagenet', include_top=True)
print(f" Model loaded  | Parameters: {model.count_params():,}")
print(f"   Input shape  : {model.input_shape}")
print(f"   Output shape : {model.output_shape}")

# Quick dummy inference test
dummy = np.zeros((1, 224, 224, 3), dtype=np.float32)
dummy = preprocess_input(dummy)
out   = model.predict(dummy, verbose=0)
print(f"   Test predict : shape={out.shape}, sum={out.sum():.4f}  ← should be ~1.0 (Softmax)")
print("\n All systems go — ready to launch Streamlit!")

Loading MobileNetV2 ...
14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
 Model loaded  | Parameters: 3,538,984
   Input shape  : (None, 224, 224, 3)
   Output shape : (None, 1000)
   Test predict : shape=(1, 1000), sum=1.0000  ← should be ~1.0 (Softmax)

 All systems go — ready to launch Streamlit!


## Launch Streamlit + Open ngrok Tunnel


In [5]:
import subprocess, time
from pyngrok import ngrok, conf

#  Optional: paste your free ngrok token here
NGROK_TOKEN = "3AsFdTFHuXkPnyjNHxJNwaZ5x25_3GRUmo9UacDG8oukBrAUS"

if NGROK_TOKEN:
    conf.get_default().auth_token = NGROK_TOKEN

# Kill any existing tunnels / streamlit processes
ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
time.sleep(1)

# Start Streamlit in background
proc = subprocess.Popen(
    ["streamlit", "run", "app.py",
     "--server.port", "8501",
     "--server.headless", "true",
     "--server.enableCORS", "false",
     "--server.enableXsrfProtection", "false"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(4)   # wait for Streamlit to boot

# Open public tunnel
public_url = ngrok.connect(8501)

print("╔══════════════════════════════════════════════════════╗")
print("║            ObjectLens AI is LIVE                   ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  URL → {str(public_url):<45}║")
print("╚══════════════════════════════════════════════════════╝")
print("\n  Open the URL above in any browser.")
print("  Use the Upload tab to test with images.")
print("  Keep this cell running — closing it kills the app.")

╔══════════════════════════════════════════════════════╗
║            ObjectLens AI is LIVE                   ║
╠══════════════════════════════════════════════════════╣
║  URL → NgrokTunnel: "https://pretheological-preconnubial-tianna.ngrok-free.dev" -> "http://localhost:8501"║
╚══════════════════════════════════════════════════════╝

  Open the URL above in any browser.
  Use the Upload tab to test with images.
  Keep this cell running — closing it kills the app.


## Stop the App

In [7]:
from pyngrok import ngrok
import subprocess

ngrok.kill()
subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)
print(" App stopped. Tunnel closed.")

 App stopped. Tunnel closed.


---
##  Technical Reference

### Why `expand_dims` is required
```
After loading image:     shape = (224, 224, 3)    ← 3D
After expand_dims:       shape = (1, 224, 224, 3) ← 4D  ✅
model.predict() needs:         (batch, H, W, C)
```
Even for a single image, Keras always expects a batch dimension at axis=0.

### Softmax in the final layer
$$\text{Softmax}(z_i) = \frac{e^{z_i}}{\sum_{j=1}^{1000} e^{z_j}}$$

- Converts raw logits → probability distribution over 1,000 classes
- All 1,000 scores sum to exactly 1.0
- Allows direct interpretation as confidence percentages

### Top-K Filtering Strategy
We search **top-20** (not just top-1) because ImageNet labels are very specific  
(e.g. `cellular_telephone`, not `phone`). The target object may rank 3rd or 5th  
even though the model clearly "sees" it.

---
*Built by Bashir Ahmad | B.Tech CSE · MNNIT Allahabad*